In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
import warnings
from tqdm import tqdm
import os
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Concrete_Strength/")

In [2]:
concrete = pd.read_csv("Concrete_Data.csv")
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [3]:
X, y = concrete.drop("Strength", axis = 1), concrete["Strength"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 26, test_size = 0.3)

In [11]:
# ss = StandardScaler()
# X_train = ss.fit_transform(X_train)
# X_test = ss.transform(X_test)

In [9]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            xgb = XGBRegressor(random_state=26, n_estimators=n, learning_rate=r, max_depth=d)
            xgb.fit(X_train, y_train)
            y_pred = xgb.predict(X_test)
            scores.append([n, r, d,r2_score(y_test, y_pred)])
            
df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "R2 Score"])
df_scores.sort_values("R2 Score", ascending = False)

100%|████████████████| 5/5 [00:31<00:00,  6.28s/it]


,Number Of Estimators,Learning Rate,Depth,R2 Score
701,100,0.620721,3,0.931011
696,100,0.589690,3,0.930562
546,75,0.589690,3,0.929912
551,75,0.620721,3,0.929805
676,100,0.465566,3,0.928821
...,...,...,...,...
4,10,0.000100,6,0.001415
3,10,0.000100,5,0.001339
2,10,0.000100,4,0.001202
1,10,0.000100,3,0.001041


In [10]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            cbm = CatBoostRegressor(random_state=26, n_estimators=n, learning_rate=r, max_depth=d,verbose=0)
            cbm.fit(X_train, y_train)
            y_pred = cbm.predict(X_test)
            scores.append([n, r, d,r2_score(y_test, y_pred)])
            
df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "R2 Score"])
df_scores.sort_values("R2 Score", ascending = False)

100%|████████████████| 5/5 [00:35<00:00,  7.01s/it]


,Number Of Estimators,Learning Rate,Depth,R2 Score
683,100,0.496597,5,0.943997
698,100,0.589690,5,0.942661
548,75,0.589690,5,0.942125
703,100,0.620721,5,0.941867
568,75,0.713814,5,0.940571
...,...,...,...,...
4,10,0.000100,6,0.000923
3,10,0.000100,5,0.000909
2,10,0.000100,4,0.000837
1,10,0.000100,3,0.000822


In [11]:
n_estimators = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []

for n in tqdm(n_estimators):
    for r in rate:
        for d in depth:
            lgbm = LGBMRegressor(random_state=26, n_estimators=n, learning_rate=r, max_depth=d,verbose=-1)
            lgbm.fit(X_train, y_train)
            y_pred = lgbm.predict(X_test)
            scores.append([n, r, d,r2_score(y_test, y_pred)])
            
df_scores = pd.DataFrame(scores, columns = ["Number Of Estimators", "Learning Rate", "Depth", "R2 Score"])
df_scores.sort_values("R2 Score", ascending = False)

100%|████████████████| 5/5 [00:09<00:00,  1.99s/it]


,Number Of Estimators,Learning Rate,Depth,R2 Score
662,100,0.372472,4,0.929793
703,100,0.620721,5,0.929716
649,100,0.279379,6,0.929664
673,100,0.434534,5,0.929581
678,100,0.465566,5,0.929403
...,...,...,...,...
4,10,0.000100,6,0.001380
3,10,0.000100,5,0.001338
2,10,0.000100,4,0.001254
1,10,0.000100,3,0.001104


### CATBoost Regressor On Housing Dataset without one hot encoding categorical variables

In [12]:
os.chdir("/home/pgcp-ai/MachineLearning/Datasets")

In [13]:
housing = pd.read_csv("Housing.csv")
housing

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...
541,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


In [20]:
housing.select_dtypes(include=object).columns.values

array(['driveway', 'recroom', 'fullbase', 'gashw', 'airco', 'prefarea'],
      dtype=object)

In [16]:
X, y = housing.drop('price',axis = 1), housing['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 26, test_size = 0.3)

In [22]:
cbm = CatBoostRegressor(random_state = 26, verbose = 0, 
                        cat_features=housing.select_dtypes(include=object).columns.values)
cbm.fit(X_train, y_train)
y_pred = cbm.predict(X_test)
r2_score(y_test, y_pred)

0.6392155092245881